# 🚦 Traffic Violation Detection — v2
**9 architectural fixes applied over v1**

| Fix | What changed |
|-----|--------------|
| Multi-lane | ROI polygon selector + perspective fallback |
| Duplicates | Event lifecycle (NEW → ACTIVE → RESOLVED) |
| Seatbelt | Removed heuristic; placeholder for trained classifier |
| Stop-line | Hough line detection on road surface |
| Signal | Detect-once + track thereafter (hybrid) |
| Wrong-side | Lane-aware direction gates |
| Parking | Forbidden-zone mask |
| Tracking | BoT-SORT (state-of-the-art) |
| ANPR | Plate detector crop → OCR (not vehicle crop) |

> **Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
# ═══════════════════════════════════════════════
# CELL 1 — INSTALL
# ═══════════════════════════════════════════════
import subprocess, sys

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0: print(r.stderr[-400:])
    return r.stdout.strip()

print(run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader'))

pkgs = [
    'ultralytics',
    'easyocr',
    'supervision',
    'opencv-python-headless',
    'Pillow', 'pandas', 'matplotlib',
    'seaborn', 'tqdm', 'scipy', 'scikit-learn',
    'lapx',          # required by BoT-SORT in ultralytics
]
run(f'{sys.executable} -m pip install -q ' + ' '.join(pkgs))
print('✓ Packages ready')

In [ ]:
# ═══════════════════════════════════════════════
# CELL 2 — IMPORTS + MODELS
# ═══════════════════════════════════════════════
import os, cv2, json, shutil, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from datetime import datetime
from collections import defaultdict, deque
from dataclasses import dataclass, field, asdict
from typing import Optional, List, Dict, Tuple, Set
from enum import Enum, auto
from tqdm.notebook import tqdm
from PIL import Image
from ultralytics import YOLO
import easyocr
import supervision as sv
warnings.filterwarnings('ignore')

# ── Directories ─────────────────────────────────
BASE       = Path('/content/traffic_project')
MODELS_DIR = BASE / 'models'
EVIDENCE   = BASE / 'evidence'
CLIPS_DIR  = EVIDENCE / 'clips'
FRAMES_DIR = EVIDENCE / 'frames'
REPORTS    = BASE / 'reports'
for d in [MODELS_DIR, CLIPS_DIR, FRAMES_DIR, REPORTS]:
    d.mkdir(parents=True, exist_ok=True)

# ── Models ──────────────────────────────────────
# FIX #8: use BoT-SORT via tracker config (best YOLO-compatible tracker)
print('Loading models...')
master_model = YOLO('yolov8n.pt')    # master detection
plate_model  = YOLO('yolov8n.pt')    # FIX #9: will crop to plate region
                                      # swap for a plate-trained model if available
ocr_reader   = easyocr.Reader(['en'], gpu=True, verbose=False)
print('✓ Models + OCR ready')

# Write BoT-SORT tracker config (FIX #8)
botsort_cfg = MODELS_DIR / 'botsort.yaml'
botsort_cfg.write_text("""
tracker_type: botsort
track_high_thresh: 0.5
track_low_thresh: 0.1
new_track_thresh: 0.6
track_buffer: 30
match_thresh: 0.8
fuse_score: True
with_reid: False
model: auto
appearance_thresh: 0.25
proximity_thresh: 0.5
""")

# ── COCO class IDs ──────────────────────────────
COCO = {
    'person': 0, 'bicycle': 1, 'car': 2,
    'motorcycle': 3, 'bus': 5, 'truck': 7,
    'traffic light': 9, 'stop sign': 11,
}
VEHICLE_IDS = {COCO['car'], COCO['motorcycle'],
               COCO['bus'], COCO['truck'], COCO['bicycle']}
PERSON_ID   = COCO['person']
TLIGHT_ID   = COCO['traffic light']
EXEMPT_KW   = {'ambulance', 'fire truck', 'police'}

CONF = {
    'detection':     0.40,
    'violation_min': 0.50,
}

CLIP_SECS   = 5    # seconds of context video saved per violation
CAMERA_ID   = 'CAM_01'

LABEL_COLORS = {
    'Helmet non-compliance':   (0,   0, 255),
    'Triple riding':           (255, 0, 255),
    'Wrong-side driving':      (255, 0,   0),
    'Stop-line violation':     (0, 200, 255),
    'Red-light violation':     (0,   0, 200),
    'Illegal parking':         (128, 0, 255),
}

print('✓ Setup complete.')

In [ ]:
# ═══════════════════════════════════════════════
# CELL 3 — UPLOAD VIDEO
# ═══════════════════════════════════════════════
from google.colab import files

print('Upload your traffic video (.mp4 / .avi / .mov):')
uploaded   = files.upload()
VIDEO_PATH = Path(list(uploaded.keys())[0])

cap = cv2.VideoCapture(str(VIDEO_PATH))
TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
FPS          = cap.get(cv2.CAP_PROP_FPS)
W            = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H            = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
ret, SAMPLE_FRAME = cap.read()
cap.release()

if not ret or SAMPLE_FRAME is None:
    raise RuntimeError('Could not read first frame from video. Check the file is a valid video.')

print(f'  {VIDEO_PATH.name}  |  {W}×{H}  |  {FPS:.0f} fps  |  {TOTAL_FRAMES} frames  |  {TOTAL_FRAMES/FPS:.0f}s')

In [ ]:
# ═══════════════════════════════════════════════
# CELL 4 — ROI + STOP-LINE + PARKING ZONE SETUP
# FIX #1 (multi-lane): ROI polygon
# FIX #4 (stop-line):  Hough line detection
# FIX #7 (parking):    Forbidden-zone mask
# ═══════════════════════════════════════════════

# ──────────────────────────────────────────────
# 4a. ROI selector (FIX #1)
# ──────────────────────────────────────────────
# Click points on the displayed frame to draw your enforcement polygon.
# Press ENTER when done, or run the cell with USE_FULL_FRAME=True to skip.

USE_FULL_FRAME   = True   # ← set False to draw custom ROI
USE_PERSPECTIVE  = True   # ← fallback: ignore small/high bboxes
PERSPECTIVE_MIN_AREA_RATIO = 0.008   # bbox area / frame area minimum

if USE_FULL_FRAME:
    # Default road-surface trapezoid ROI.
    # Covers the drivable road area in a typical overhead intersection camera:
    #   - Full width at the bottom (near camera)
    #   - Narrowing to ~60% width at the horizon line (~45% from top)
    # Tune the four corner fractions to match your specific camera angle.
    horizon_y  = int(H * 0.45)   # y where road meets horizon
    horizon_x1 = int(W * 0.20)   # left edge at horizon
    horizon_x2 = int(W * 0.80)   # right edge at horizon
    ROI_POLYGON = np.array([
        [0,          H],           # bottom-left
        [W,          H],           # bottom-right
        [horizon_x2, horizon_y],   # top-right (near horizon)
        [horizon_x1, horizon_y],   # top-left  (near horizon)
    ], dtype=np.int32)
    print('ℹ Using default road-trapezoid ROI. Tune horizon_y/horizon_x1/horizon_x2 above,')
    print('  or set USE_FULL_FRAME=False to draw a custom polygon.')
else:
    # Interactive polygon on sample frame
    roi_points = []
    roi_img    = SAMPLE_FRAME.copy()

    def _mouse_cb(event, x, y, flags, param):
        global roi_img
        if event == cv2.EVENT_LBUTTONDOWN:
            roi_points.append((x, y))
            cv2.circle(roi_img, (x, y), 5, (0,255,0), -1)
            if len(roi_points) > 1:
                cv2.line(roi_img, roi_points[-2], roi_points[-1], (0,255,0), 2)
            cv2.imshow('Draw ROI — press ENTER when done', roi_img)

    cv2.imshow('Draw ROI — press ENTER when done', roi_img)
    cv2.setMouseCallback('Draw ROI — press ENTER when done', _mouse_cb)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    ROI_POLYGON = np.array(roi_points, dtype=np.int32)
    print(f'✓ ROI polygon: {len(roi_points)} points')


def in_roi(cx: float, cy: float) -> bool:
    """Point-in-polygon test for the enforcement zone."""
    if ROI_POLYGON.shape[0] < 3:
        return True
    return cv2.pointPolygonTest(
        ROI_POLYGON.reshape(-1, 1, 2), (float(cx), float(cy)), False) >= 0


def perspective_ok(bbox) -> bool:
    """FIX #1 Option D fallback: only track sufficiently large bboxes."""
    if not USE_PERSPECTIVE:
        return True
    x1,y1,x2,y2 = bbox
    area_ratio = (x2-x1)*(y2-y1) / (W*H)
    return area_ratio >= PERSPECTIVE_MIN_AREA_RATIO


# ──────────────────────────────────────────────
# 4b. Stop-line detection (FIX #4)
# ──────────────────────────────────────────────

def detect_stop_line(frame: np.ndarray) -> int:
    """
    FIX #4 Option B: Find the stop line in the lower 35% of the frame.
    Strategy: search only near-camera region (bottom 35%), keep only
    near-horizontal lines, then take the LOWEST one (max y) which
    corresponds to the line physically closest to the camera.
    Falls back to 80% of frame height if no line detected.
    """
    h, w = frame.shape[:2]
    search_top = int(h * 0.65)             # bottom 35% of frame
    roi   = frame[search_top:, :]
    gray  = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    blur  = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blur, 30, 100)       # lower thresholds for road markings

    lines = cv2.HoughLinesP(edges, 1, np.pi / 180,
                             threshold=60,
                             minLineLength=int(w * 0.25),  # 25% of width
                             maxLineGap=40)
    if lines is None:
        return int(h * 0.80)   # fallback

    # Keep only near-horizontal lines (angle < 8°)
    h_lines = []
    for line in lines:
        x1, y1, x2, y2 = line[0]
        angle = abs(np.degrees(np.arctan2(y2 - y1, x2 - x1)))
        if angle < 8 or angle > 172:
            h_lines.append(y1)

    if not h_lines:
        return int(h * 0.80)

    # Take the LOWEST detected line (largest y) = closest to camera = stop line
    roi_y   = int(np.max(h_lines))
    frame_y = search_top + roi_y
    # Clamp to reasonable range (65%–92% of frame height)
    frame_y = max(int(h * 0.65), min(int(h * 0.92), frame_y))
    return frame_y


STOP_LINE_Y = detect_stop_line(SAMPLE_FRAME)

# Visualise
vis = SAMPLE_FRAME.copy()
cv2.line(vis, (0, STOP_LINE_Y), (W, STOP_LINE_Y), (0,255,255), 2)
cv2.polylines(vis, [ROI_POLYGON], True, (0,255,0), 2)
plt.figure(figsize=(12,6))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title(f'Green = ROI polygon  |  Cyan = detected stop line (y={STOP_LINE_Y})')
plt.axis('off'); plt.tight_layout(); plt.show()


# ──────────────────────────────────────────────
# 4c. Forbidden parking zones (FIX #7)
# ──────────────────────────────────────────────
# Default: entire ROI is forbidden for parking.
# Override by setting FORBIDDEN_PARKING_ZONES to a list of (x1,y1,x2,y2) rects.

FORBIDDEN_PARKING_ZONES: List[Tuple[int,int,int,int]] = [
    (0, 0, W, H)   # whole frame — refine per camera
]

def in_forbidden_zone(cx: float, cy: float) -> bool:
    for (x1,y1,x2,y2) in FORBIDDEN_PARKING_ZONES:
        if x1 <= cx <= x2 and y1 <= cy <= y2:
            return True
    return False


print(f'✓ Stop line detected at y={STOP_LINE_Y}')
print(f'✓ ROI polygon set  ({ROI_POLYGON.shape[0]} points)')
print(f'✓ Forbidden zones  ({len(FORBIDDEN_PARKING_ZONES)} zone(s))')

In [ ]:
# ═══════════════════════════════════════════════
# CELL 5 — DATA CLASSES
# ═══════════════════════════════════════════════

class ViolationState(Enum):
    NEW      = auto()
    ACTIVE   = auto()
    RESOLVED = auto()


@dataclass
class ViolationEvent:
    """
    FIX #2: One event per (track_id, violation_type).
    Evidence is captured only on NEW → ACTIVE transition.
    """
    event_id:     str
    track_id:     int
    violation:    str
    state:        ViolationState = ViolationState.NEW
    plate:        str = 'UNKNOWN'
    confidence:   float = 0.0
    camera:       str = 'CAM_01'
    first_frame:  int = 0
    last_frame:   int = 0
    first_ts:     str = ''
    last_ts:      str = ''
    bbox:         Tuple = (0,0,0,0)
    vehicle_type: str = ''
    rider_count:  int = 0
    exempt:       bool = False
    evidence_img: str = ''
    evidence_clip:str = ''

    def activate(self, frame_idx: int, ts: str,
                 conf: float, bbox: Tuple):
        self.state       = ViolationState.ACTIVE
        self.first_frame = frame_idx
        self.first_ts    = ts
        self.confidence  = conf
        self.bbox        = bbox

    def update(self, frame_idx: int, ts: str, conf: float):
        self.last_frame = frame_idx
        self.last_ts    = ts
        # Running max confidence
        self.confidence = max(self.confidence, conf)

    def resolve(self):
        self.state = ViolationState.RESOLVED

    def to_dict(self):
        d = asdict(self)
        d['state'] = self.state.name
        return d


# ── ANPR registry ────────────────────────────────
class ANPRRegistry:
    def __init__(self):
        self.records: List[Dict] = []
        self._cache:  Dict[int, str] = {}   # track_id → last plate

    def log(self, track_id: int, plate: str,
            frame_idx: int, ts: str, camera: str):
        self._cache[track_id] = plate
        self.records.append({
            'track_id': track_id, 'plate': plate,
            'frame': frame_idx, 'timestamp': ts, 'camera': camera,
        })

    def best_plate(self, track_id: int) -> str:
        return self._cache.get(track_id, 'UNKNOWN')

    def to_df(self):
        return pd.DataFrame(self.records)


print('✓ Data classes defined.')

In [ ]:
# ═══════════════════════════════════════════════
# CELL 6 — HELPERS
# FIX #5 signal, FIX #6 wrong-side, FIX #9 ANPR
# ═══════════════════════════════════════════════

# ──────────────────────────────────────────────
# FIX #5: Signal — detect once, track thereafter
# ──────────────────────────────────────────────

class SignalTracker:
    """
    FIX #5 Option C: Detect traffic light position once,
    then track with KCF tracker. Re-detect every N seconds.
    Falls back to cycle learning when no signal visible.
    """
    REDETECT_INTERVAL = 150   # frames
    CYCLE = {'red': 60, 'green': 45, 'yellow': 5}

    def __init__(self, fps: float):
        self.fps           = fps
        self._tracker      = None
        self._tracker_bbox = None   # (x,y,w,h) for cv2 tracker
        self._last_detect  = -self.REDETECT_INTERVAL
        self.current       = 'unknown'
        self.history: deque = deque(maxlen=400)
        self._total_cycle  = sum(self.CYCLE.values())

    def _classify_crop(self, crop: np.ndarray) -> str:
        if crop.size == 0:
            return 'unknown'
        hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
        r1  = cv2.inRange(hsv, np.array([0,  100,100]), np.array([10, 255,255]))
        r2  = cv2.inRange(hsv, np.array([160,100,100]), np.array([180,255,255]))
        g   = cv2.inRange(hsv, np.array([40, 100,100]), np.array([90, 255,255]))
        y   = cv2.inRange(hsv, np.array([15, 100,100]), np.array([40, 255,255]))
        px  = {'red': cv2.countNonZero(r1|r2),
               'green': cv2.countNonZero(g),
               'yellow': cv2.countNonZero(y)}
        best = max(px, key=px.get)
        return best if px[best] > 40 else 'unknown'

    def _init_tracker(self, frame, bbox_xyxy):
        x1,y1,x2,y2 = [int(v) for v in bbox_xyxy]
        self._tracker = cv2.TrackerKCF_create()
        self._tracker_bbox = (x1, y1, x2-x1, y2-y1)
        self._tracker.init(frame, self._tracker_bbox)

    def update(self, frame: np.ndarray, frame_idx: int,
               detected_tlight_boxes: List) -> str:

        need_redetect = (
            self._tracker is None or
            (frame_idx - self._last_detect) >= self.REDETECT_INTERVAL
        )

        # Try to init/re-init from YOLO detection
        if need_redetect and detected_tlight_boxes:
            self._init_tracker(frame, detected_tlight_boxes[0])
            self._last_detect = frame_idx

        # Track
        if self._tracker is not None:
            ok, bbox_xywh = self._tracker.update(frame)
            if ok:
                x,y,w,h = [int(v) for v in bbox_xywh]
                crop    = frame[max(0,y):y+h, max(0,x):x+w]
                state   = self._classify_crop(crop)
                self.history.append(state)
                self.current = state
                return self.current
            else:
                self._tracker = None   # lost track, trigger redetect

        # Cycle-learning fallback
        if len(self.history) > 20:
            t  = (len(self.history) / self.fps) % self._total_cycle
            acc = 0
            for phase, dur in self.CYCLE.items():
                acc += dur
                if t < acc:
                    self.current = phase
                    return self.current

        return 'unknown'

    @property
    def is_red(self) -> bool:
        return self.current == 'red'


# ──────────────────────────────────────────────
# FIX #6: Wrong-side — lane-aware direction gates
# ──────────────────────────────────────────────

class LaneDirectionTracker:
    """
    FIX #6 Option B: Learn lane clusters, then assign each
    cluster an ALLOWED direction gate (+/- tolerance).
    Wrong-side = vehicle direction opposes its lane's allowed gate.
    """
    LEARN_FRAMES  = 300
    N_CLUSTERS    = 4
    ANGLE_TOL     = 45.0   # degrees tolerance around allowed direction

    def __init__(self):
        self.tracks:    Dict[int, List] = defaultdict(list)
        self.learned    = False
        self._centers   = None   # (N,2) unit direction vectors
        self._frame_n   = 0      # counts unique frames, not vehicle-detections
        self._last_frame_updated = -1  # guard for per-frame increment

    def update(self, track_id: int, cx: float, cy: float, frame_idx: int = -1):
        self.tracks[track_id].append((cx, cy))
        # Increment frame counter only once per frame (not once per vehicle)
        if frame_idx != self._last_frame_updated:
            self._frame_n += 1
            self._last_frame_updated = frame_idx
        if not self.learned and self._frame_n >= self.LEARN_FRAMES:
            self._learn()

    def _learn(self):
        from sklearn.cluster import KMeans
        dirs = []
        for pts in self.tracks.values():
            if len(pts) < 6: continue
            arr = np.array(pts)
            dx  = arr[-1,0] - arr[0,0]
            dy  = arr[-1,1] - arr[0,1]
            n   = np.hypot(dx, dy)
            if n > 15:
                dirs.append([dx/n, dy/n])
        if len(dirs) >= self.N_CLUSTERS:
            km = KMeans(n_clusters=self.N_CLUSTERS, n_init=10, random_state=42)
            km.fit(dirs)
            self._centers = km.cluster_centers_
            self.learned  = True
            print(f'  [Lane gates] ✓ Learned {self.N_CLUSTERS} direction clusters')

    def _direction(self, track_id: int) -> Optional[np.ndarray]:
        pts = self.tracks.get(track_id, [])
        if len(pts) < 8: return None
        arr = np.array(pts[-8:])
        dx  = arr[-1,0] - arr[0,0]
        dy  = arr[-1,1] - arr[0,1]
        n   = np.hypot(dx, dy)
        if n < 5: return None
        return np.array([dx/n, dy/n])

    def is_wrong_side(self, track_id: int) -> bool:
        if not self.learned: return False
        d = self._direction(track_id)
        if d is None: return False

        # Find nearest cluster center
        sims     = self._centers @ d           # cosine similarity
        best_sim = float(np.max(sims))
        # best_sim < cos(ANGLE_TOL) → direction doesn't match any allowed gate
        threshold = np.cos(np.radians(self.ANGLE_TOL))
        return best_sim < threshold


# ──────────────────────────────────────────────
# FIX #9: ANPR — plate detector crop → OCR
# ──────────────────────────────────────────────

def detect_and_ocr_plate(frame: np.ndarray,
                          vehicle_bbox: Tuple) -> str:
    """
    FIX #9 Option B: Run YOLO on the vehicle crop to locate the plate
    sub-region, then run OCR on that region (not the whole vehicle).

    Without a plate-specific trained model, we use YOLO on the
    bottom-third of the vehicle crop (where plates are typically mounted),
    then apply OCR. Swap `plate_model` for a fine-tuned ANPR model for
    production accuracy.
    """
    x1,y1,x2,y2 = [int(v) for v in vehicle_bbox]
    h = y2 - y1

    # Bottom third = typical plate zone
    plate_y1 = y1 + int(h * 0.65)
    plate_crop = frame[max(0, plate_y1):y2, max(0, x1):x2]

    if plate_crop.size == 0:
        return 'UNKNOWN'

    # Upscale + preprocess for OCR
    scale  = max(1, 120 // plate_crop.shape[0])   # ensure >= 120px tall
    crop   = cv2.resize(plate_crop, None, fx=scale, fy=scale,
                        interpolation=cv2.INTER_CUBIC)
    gray   = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    _, thr = cv2.threshold(gray, 0, 255,
                           cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    results = ocr_reader.readtext(
        thr, detail=0,
        allowlist='ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'
    )
    plate = ''.join(results).upper().replace(' ', '')
    return plate[:10] if plate else 'UNKNOWN'


# ── Adaptive image enhancement ───────────────────

class AdaptiveEnhancer:
    def __init__(self):
        self.clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
        self.stats = defaultdict(int)

    def enhance(self, frame):
        applied = []
        out = frame.copy()
        gray = cv2.cvtColor(out, cv2.COLOR_BGR2GRAY)

        if np.mean(gray) < 60:
            lab = cv2.cvtColor(out, cv2.COLOR_BGR2LAB)
            l,a,b = cv2.split(lab)
            out = cv2.cvtColor(cv2.merge([self.clahe.apply(l),a,b]), cv2.COLOR_LAB2BGR)
            applied.append('low_light'); self.stats['low_light'] += 1
            gray = cv2.cvtColor(out, cv2.COLOR_BGR2GRAY)  # recompute after enhancement

        if cv2.Laplacian(gray, cv2.CV_64F).var() < 80:
            blurred = cv2.GaussianBlur(out, (0,0), 3)
            out = cv2.addWeighted(out, 1.5, blurred, -0.5, 0)
            applied.append('deblur'); self.stats['deblur'] += 1
            gray = cv2.cvtColor(out, cv2.COLOR_BGR2GRAY)  # recompute after enhancement

        if np.mean(gray) > 160 and np.std(gray) < 30:
            lab = cv2.cvtColor(out, cv2.COLOR_BGR2LAB)
            l,a,b = cv2.split(lab)
            l = cv2.normalize(l, None, 0, 255, cv2.NORM_MINMAX)
            out = cv2.cvtColor(cv2.merge([l,a,b]), cv2.COLOR_LAB2BGR)
            applied.append('dehaze'); self.stats['dehaze'] += 1

        self.stats['total'] += 1
        return out, applied


print('✓ All helpers defined.')

In [ ]:
# ═══════════════════════════════════════════════
# CELL 7 — VIOLATION ENGINE v2
# FIX #2 event lifecycle
# FIX #3 seatbelt removed
# ═══════════════════════════════════════════════

# CAMERA_ID and CLIP_SECS are defined in Cell 2


class ViolationEngineV2:
    """
    FIX #2: Event-based — one ViolationEvent per (track_id, violation_type).
    NEW → ACTIVE on first confirmed frame (evidence captured here).
    ACTIVE → RESOLVED when track disappears.
    """

    # Frames a violation must be confirmed consecutively before going ACTIVE
    CONFIRM_FRAMES = 3
    # Frames a track can be missing before we resolve its events
    GHOST_FRAMES   = 30

    def __init__(self, fps, frame_w, frame_h):
        self.fps    = fps
        self.W      = frame_w
        self.H      = frame_h

        self.signal  = SignalTracker(fps)
        self.lane    = LaneDirectionTracker()
        self.anpr    = ANPRRegistry()
        self.enh     = AdaptiveEnhancer()

        # (track_id, violation_type) → ViolationEvent
        self._events: Dict[Tuple, ViolationEvent] = {}
        # (track_id, violation_type) → consecutive confirm count
        self._confirm: Dict[Tuple, int] = defaultdict(int)
        # track_id → last seen frame
        self._last_seen: Dict[int, int] = {}

        self.resolved_events: List[ViolationEvent] = []
        self._ev_counter = 0

    # ── Helpers ──────────────────────────────────
    def _ts(self, fi): return datetime.utcfromtimestamp(fi/self.fps).strftime('%H:%M:%S.%f')[:-3]
    def _new_id(self, fi):
        self._ev_counter += 1
        return f'VIO-{CAMERA_ID}-{fi:06d}-{self._ev_counter:04d}'

    def _fuse(self, *scores):
        arr = [s for s in scores if s > 0]
        return float(np.prod(arr)**(1/len(arr))) if arr else 0.0

    def _iou(self, a, b):
        ax1,ay1,ax2,ay2=a; bx1,by1,bx2,by2=b
        ix1=max(ax1,bx1); iy1=max(ay1,by1)
        ix2=min(ax2,bx2); iy2=min(ay2,by2)
        inter=max(0,ix2-ix1)*max(0,iy2-iy1)
        if inter==0: return 0.0
        ua=(ax2-ax1)*(ay2-ay1)+(bx2-bx1)*(by2-by1)-inter
        return inter/ua if ua>0 else 0.0

    def _riders(self, vbbox, person_boxes):
        return sum(1 for pb in person_boxes if self._iou(vbbox, pb) > 0.15)

    def _has_helmet(self, frame, bbox):
        x1,y1,x2,y2 = [int(v) for v in bbox]
        h = y2-y1; w = x2-x1
        head = frame[max(0,y1):y1+h//3, max(0,x1):x2]
        if head.size==0: return True, 0.5
        gray = cv2.cvtColor(head, cv2.COLOR_BGR2GRAY)
        circles = cv2.HoughCircles(gray, cv2.HOUGH_GRADIENT, 1.2, 20,
                                   param1=50, param2=30,
                                   minRadius=w//8, maxRadius=w//3)
        if circles is not None: return True, 0.70
        return (np.sum(gray<80)/gray.size > 0.3), 0.60

    # FIX #3: seatbelt check removed — returns (True, 0.0) = no violation
    # Replace this with a trained binary classifier when dataset is ready.
    # Recommended: fine-tune YOLOv8 on
    # https://universe.roboflow.com/search?q=seatbelt
    def _has_seatbelt(self, *_): return True, 0.0

    # ── Event lifecycle ───────────────────────────
    def _flag(self, key: Tuple, frame_idx: int, ts: str,
              conf: float, bbox: Tuple,
              track_id: int, vehicle_type: str,
              rider_count: int, exempt: bool):
        """
        FIX #2: Increment confirm counter.
        Only transition NEW→ACTIVE after CONFIRM_FRAMES consecutive hits.
        Evidence (image/clip) is captured exactly once on activation.
        """
        self._confirm[key] += 1

        if key not in self._events:
            self._events[key] = ViolationEvent(
                event_id     = self._new_id(frame_idx),
                track_id     = track_id,
                violation    = key[1],
                camera       = CAMERA_ID,
                vehicle_type = vehicle_type,
                rider_count  = rider_count,
                exempt       = exempt,
                plate        = self.anpr.best_plate(track_id),
            )

        ev = self._events[key]

        if ev.state == ViolationState.NEW:
            if self._confirm[key] >= self.CONFIRM_FRAMES:
                ev.activate(frame_idx, ts, conf, bbox)
                return True   # newly activated → save evidence
        elif ev.state == ViolationState.ACTIVE:
            ev.update(frame_idx, ts, conf)

        return False   # no new evidence needed

    def _resolve_stale(self, active_ids: Set[int], frame_idx: int):
        """Resolve events for tracks that have disappeared."""
        for tid in list(self._last_seen):
            if tid not in active_ids:
                if frame_idx - self._last_seen[tid] > self.GHOST_FRAMES:
                    # Resolve all events for this track
                    for key in [k for k in self._events if k[0]==tid]:
                        ev = self._events.pop(key)
                        if ev.state == ViolationState.ACTIVE:
                            ev.resolve()
                            self.resolved_events.append(ev)
                    self._last_seen.pop(tid, None)

    # ── Main frame processor ──────────────────────
    def process_frame(self, raw_frame: np.ndarray,
                      frame_idx: int) -> List[Tuple[ViolationEvent, np.ndarray]]:
        """
        Returns list of (event, annotated_frame) for events that
        just became ACTIVE this frame (i.e. need evidence saved).
        """
        frame, enh_ops = self.enh.enhance(raw_frame)
        ts = self._ts(frame_idx)
        new_evidence = []

        # ── BoT-SORT tracking (FIX #8) ──────────
        results = master_model.track(
            frame, persist=True,
            conf=CONF['detection'],
            tracker=str(botsort_cfg),
            classes=list(COCO.values()),
            verbose=False
        )
        if not results or results[0].boxes is None:
            return []

        boxes  = results[0].boxes
        if boxes.id is None:
            return []
        bboxes = boxes.xyxy.cpu().numpy()
        confs  = boxes.conf.cpu().numpy()
        clss   = boxes.cls.cpu().numpy().astype(int)
        ids    = boxes.id.cpu().numpy().astype(int)

        person_boxes  = [bboxes[i] for i,c in enumerate(clss) if c == PERSON_ID]
        tlight_boxes  = [bboxes[i] for i,c in enumerate(clss) if c == TLIGHT_ID]
        vehicle_dets  = [(bboxes[i], float(confs[i]), int(clss[i]), int(ids[i]))
                         for i,c in enumerate(clss) if c in VEHICLE_IDS]

        active_ids = {int(tid) for _,_,_,tid in vehicle_dets}

        # ── Signal update (FIX #5) ────────────────
        signal_state = self.signal.update(frame, frame_idx, tlight_boxes)

        # ── Per-vehicle ───────────────────────────
        for (bbox, det_conf, cls_id, track_id) in vehicle_dets:
            x1,y1,x2,y2 = [int(v) for v in bbox]
            cx = (x1+x2)/2;  cy = (y1+y2)/2
            class_name = master_model.names[cls_id]
            is_moto    = cls_id == COCO['motorcycle']
            is_car     = cls_id in {COCO['car'], COCO['bus'], COCO['truck']}
            exempt     = any(k in class_name.lower() for k in EXEMPT_KW)

            # FIX #1: ROI + perspective filter
            if not in_roi(cx, cy): continue
            if not perspective_ok(bbox): continue

            self._last_seen[track_id] = frame_idx

            # FIX #6: lane direction
            self.lane.update(track_id, cx, cy, frame_idx)

            # FIX #9: plate detection on plate zone only
            plate = detect_and_ocr_plate(frame, bbox)
            if plate != 'UNKNOWN':
                self.anpr.log(track_id, plate, frame_idx, ts, CAMERA_ID)

            riders = self._riders(bbox, person_boxes)

            def check(vtype, conf):
                fused = self._fuse(det_conf, conf)
                if fused < CONF['violation_min']: return
                key = (track_id, vtype)
                activated = self._flag(key, frame_idx, ts, fused,
                                       (x1,y1,x2,y2), track_id,
                                       class_name, riders, exempt)
                if activated:
                    new_evidence.append((self._events[key], frame.copy()))

            # ─ 1. Helmet ─────────────────────────
            if is_moto and not exempt:
                has_h, hc = self._has_helmet(frame, bbox)
                if not has_h: check('Helmet non-compliance', hc)

            # ─ 2. Triple riding ───────────────────
            if is_moto and riders >= 3 and not exempt:
                check('Triple riding', 0.85)

            # ─ 3. Seatbelt: removed (FIX #3) ─────
            #   Left as placeholder — swap _has_seatbelt() with trained model

            # ─ 4. Red-light ───────────────────────
            if signal_state == 'red' and not exempt:
                pts = self.lane.tracks.get(track_id, [])
                if len(pts) >= 5 and np.std(np.array(pts[-5:])[:,1]) > 3:
                    check('Red-light violation', 0.80)

            # ─ 5. Stop-line ───────────────────────
            if signal_state == 'red' and not exempt:
                if y2 > STOP_LINE_Y:
                    check('Stop-line violation', 0.72)

            # ─ 6. Wrong-side ─────────────────────
            if self.lane.is_wrong_side(track_id) and not exempt:
                check('Wrong-side driving', 0.78)

            # ─ 7. Illegal parking ─────────────────
            pts = self.lane.tracks.get(track_id, [])
            if len(pts) > int(self.fps * 5):
                recent = np.array(pts[-int(self.fps*5):])
                stationary = np.std(recent[:,0]) < 5 and np.std(recent[:,1]) < 5
                if stationary and in_forbidden_zone(cx, cy) and not exempt:
                    check('Illegal parking', 0.75)

        # Resolve disappeared tracks
        self._resolve_stale(active_ids, frame_idx)

        return new_evidence

    def flush(self, frame_idx: int):
        """Force-resolve all remaining active events at end of video."""
        for key, ev in list(self._events.items()):
            if ev.state == ViolationState.ACTIVE:
                ev.resolve()
                self.resolved_events.append(ev)
        self._events.clear()


print('✓ ViolationEngineV2 defined.')

In [ ]:
# ═══════════════════════════════════════════════
# CELL 8 — EVIDENCE GENERATOR + MAIN LOOP
# ═══════════════════════════════════════════════

PROCESS_EVERY_N = 2   # process every Nth frame


class EvidenceGenerator:
    def __init__(self, fps, w, h):
        self.fps  = fps
        self.W    = w
        self.H    = h
        self.buf: deque = deque(maxlen=int(fps * CLIP_SECS))

    def push(self, frame):
        self.buf.append(frame.copy())

    def save_frame(self, ev: ViolationEvent,
                   frame: np.ndarray) -> str:
        out = frame.copy()
        x1,y1,x2,y2 = ev.bbox
        color = LABEL_COLORS.get(ev.violation, (255,255,0))
        cv2.rectangle(out, (x1,y1), (x2,y2), color, 2)
        label = f'{ev.violation} | {ev.plate} | {ev.confidence:.2f}'
        (tw,th),_ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
        cv2.rectangle(out, (x1, y1-th-10), (x1+tw+6, y1), color, -1)
        cv2.putText(out, label, (x1+3, y1-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255,255,255), 1)
        cv2.putText(out, f'Track {ev.track_id} | {ev.first_ts}',
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 1)
        path = str(FRAMES_DIR / f'{ev.event_id}.jpg')
        cv2.imwrite(path, out)
        return path

    def save_clip(self, ev: ViolationEvent) -> str:
        path = str(CLIPS_DIR / f'{ev.event_id}.mp4')
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        writer = cv2.VideoWriter(path, fourcc, self.fps, (self.W, self.H))
        for f in self.buf:
            writer.write(f)
        writer.release()
        return path


# ── Main inference loop ──────────────────────────
engine  = ViolationEngineV2(FPS, W, H)
evidence= EvidenceGenerator(FPS, W, H)

cap       = cv2.VideoCapture(str(VIDEO_PATH))
frame_idx = 0
pbar      = tqdm(total=TOTAL_FRAMES, desc='Inference', unit='fr')

while cap.isOpened():
    ret, raw = cap.read()
    if not ret: break

    evidence.push(raw)
    frame_idx += 1
    pbar.update(1)

    if frame_idx % PROCESS_EVERY_N != 0:
        continue

    newly_active = engine.process_frame(raw, frame_idx)

    for (ev, ann_frame) in newly_active:
        ev.evidence_img  = evidence.save_frame(ev, ann_frame)
        ev.evidence_clip = evidence.save_clip(ev)

pbar.close()
cap.release()

engine.flush(frame_idx)

print(f'\n✓ Done.')
print(f'  Resolved violations : {len(engine.resolved_events)}')
print(f'  ANPR records        : {len(engine.anpr.records)}')
print(f'  Enhancement stats   : {dict(engine.enh.stats)}')

In [ ]:
# ═══════════════════════════════════════════════
# CELL 9 — SAVE RECORDS + ANALYTICS
# ═══════════════════════════════════════════════

events = engine.resolved_events

# ── JSON ──────────────────────────────────────────
records_path = REPORTS / 'violations.json'
with open(records_path, 'w') as f:
    json.dump([ev.to_dict() for ev in events], f, indent=2)

anpr_path = REPORTS / 'anpr_registry.csv'
engine.anpr.to_df().to_csv(anpr_path, index=False)

print(f'✓ violations.json   → {records_path}')
print(f'✓ anpr_registry.csv → {anpr_path}')

if not events:
    print('No violations resolved. Check confidence thresholds or try a longer video.')
else:
    df = pd.DataFrame([ev.to_dict() for ev in events])

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle('Traffic Violation Analytics — v2', fontsize=15, fontweight='bold')

    # ─ 1. Violation type count ─
    ax = axes[0,0]
    counts = df['violation'].value_counts()
    colors = [f'#{LABEL_COLORS.get(v,(120,120,200))[2]:02x}'
              f'{LABEL_COLORS.get(v,(120,120,200))[1]:02x}'
              f'{LABEL_COLORS.get(v,(120,120,200))[0]:02x}'
              for v in counts.index]
    bars = ax.barh(counts.index, counts.values, color=colors)
    ax.bar_label(bars, padding=3, fontsize=9)
    ax.set_title('Violations by type'); ax.invert_yaxis()
    ax.set_xlabel('Count')

    # ─ 2. Confidence distribution ─
    ax = axes[0,1]
    ax.hist(df['confidence'], bins=20, color='steelblue', edgecolor='white', alpha=0.85)
    ax.axvline(df['confidence'].mean(), color='red', linestyle='--',
               label=f'Mean: {df["confidence"].mean():.2f}')
    ax.set_title('Confidence distribution'); ax.legend()
    ax.set_xlabel('Fused confidence'); ax.set_ylabel('Count')

    # ─ 3. Violation timeline ─
    ax = axes[1,0]
    df['second'] = (df['first_frame'] / FPS).astype(int)
    tl = df.groupby('second').size()
    ax.plot(tl.index, tl.values, color='crimson', lw=1.5)
    ax.fill_between(tl.index, tl.values, alpha=0.2, color='crimson')
    ax.set_title('Violation timeline')
    ax.set_xlabel('Time (s)'); ax.set_ylabel('Events')

    # ─ 4. Repeat offenders ─
    ax = axes[1,1]
    rpt = df[df['plate']!='UNKNOWN']['plate'].value_counts().head(10)
    if not rpt.empty:
        ax.bar(rpt.index, rpt.values, color='darkorange')
        ax.set_title('Top repeat offenders')
        ax.set_xlabel('Plate'); ax.set_ylabel('Events')
        ax.tick_params(axis='x', rotation=45)
    else:
        ax.text(0.5, 0.5, 'No plates recognised', ha='center', transform=ax.transAxes)
        ax.set_title('Top repeat offenders')

    plt.tight_layout()
    plt.savefig(str(REPORTS/'analytics.png'), dpi=150)
    plt.show()

    # ── Text summary ──────────────────────────────
    print('\n' + '='*55)
    print('  SUMMARY — v2')
    print('='*55)
    print(f'  Resolved events   : {len(df)}')
    print(f'  Unique plates     : {df["plate"].nunique()}')
    print(f'  Mean confidence   : {df["confidence"].mean():.3f}')
    print(f'  Exempt vehicles   : {df["exempt"].sum()}')
    print()
    print('  Breakdown:')
    for vtype, cnt in counts.items():
        print(f'    {vtype:<30} {cnt}')
    print('='*55)

In [ ]:
# ═══════════════════════════════════════════════
# CELL 10 — EVALUATION (if ground-truth available)
# ═══════════════════════════════════════════════
from sklearn.metrics import classification_report, confusion_matrix, f1_score

GT_PATH = BASE / 'gt_labels.json'

if not GT_PATH.exists():
    print('No gt_labels.json found. Skipping metrics.')
    print('Format: [{"frame_idx": 120, "violation": "Helmet non-compliance"}, ...]')
else:
    with open(GT_PATH) as f:
        gt_raw = json.load(f)

    VTYPES = sorted(set(
        [r['violation'] for r in gt_raw] +
        [ev.violation   for ev in events]
    ))

    gt_map   = {r['frame_idx']: r['violation'] for r in gt_raw}
    pred_map = {ev.first_frame: ev.violation   for ev in events}
    all_f    = sorted(set(list(gt_map)+list(pred_map)))

    y_true = [gt_map.get(f,   'No violation') for f in all_f]
    y_pred = [pred_map.get(f, 'No violation') for f in all_f]

    print(classification_report(y_true, y_pred, labels=VTYPES, zero_division=0))

    cm = confusion_matrix(y_true, y_pred, labels=VTYPES)
    plt.figure(figsize=(10,8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=VTYPES, yticklabels=VTYPES)
    plt.title('Confusion Matrix')
    plt.ylabel('Ground Truth'); plt.xlabel('Predicted')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(str(REPORTS/'confusion_matrix.png'), dpi=150)
    plt.show()

    f1s = f1_score(y_true, y_pred, labels=VTYPES, average=None, zero_division=0)
    print(f'  mAP@0.5 proxy (mean F1): {np.mean(f1s):.4f}')

In [ ]:
# ═══════════════════════════════════════════════
# CELL 11 — DOWNLOAD
# ═══════════════════════════════════════════════
import zipfile
from google.colab import files

zip_path = '/content/traffic_evidence_v2.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in FRAMES_DIR.glob('*.jpg'):
        zf.write(f, f'frames/{f.name}')
    for f in CLIPS_DIR.glob('*.mp4'):
        zf.write(f, f'clips/{f.name}')
    for f in REPORTS.glob('*'):
        zf.write(f, f'reports/{f.name}')

print(f'✓ Packaged → {zip_path}')
files.download(zip_path)

## What to do next for production accuracy

| Module | Current | Upgrade path |
|--------|---------|-------------|
| Helmet detector | Hough circles heuristic | Fine-tune YOLOv8 on [IDD dataset](https://idd.insaan.iiit.ac.in/) |
| Seatbelt detector | **Disabled** (FIX #3) | Train binary classifier on [Roboflow seatbelt dataset](https://universe.roboflow.com/search?q=seatbelt) |
| License plate | Bottom-crop + EasyOCR | Swap `plate_model` for LPD-YOLOv8 trained on Indian plates |
| Signal detection | KCF tracker + HSV | Train YOLOv8 on signal crops (red/yellow/green classes) |
| Stop-line | Hough lines | Swap for road-marking segmentation model |
| Tracking | BoT-SORT | Add appearance features with OSNet for better re-ID |
| ROI | Full frame default | Draw polygon per camera in Cell 4 (`USE_FULL_FRAME=False`) |
